# 10.1 - Evaluation - Benchmarking Models and Agent Configurations with AIMU

This notebook uses AIMU's [`Benchmark`](https://github.com/InstituteforDiseaseModeling/aimu) harness to compare multiple model clients and agent configurations across two scientific tasks relevant to this project:

1. **Disease modeling classification** - given an abstract, decide whether it describes a disease modeling study (the project's core paper-triage task). Scored with a custom exact-match `Scorer` against the labeled training set.
2. **Scientific abstract summarization** - condense an abstract into a plain-English methods-and-findings summary. Scored with an `LLMJudgeScorer` using a stronger model as judge.

The same `Benchmark` is run against a slate of model clients including:

- One or more local Ollama models.
- A frontier Anthropic model.
- An `AgenticModelClient` that wraps a `SimpleAgent` with access to the genscai MCP `search_research_articles` tool, so the agent can ground its summaries in retrieved literature.

`Benchmark` runs each row through `client.chat()` with messages reset between rows, so the agentic client engages its full tool-use loop on every example. Aggregate `score` and `pass_rate` per client are returned as a DataFrame.

## 01. Setup

Imports, paths, and a check that the labeled classification dataset and the medRxiv vector database (used by the agent's MCP tool) are present.

In [ ]:
import pandas as pd
import nest_asyncio

from genscai import paths, data

from aimu.evals import Benchmark
from aimu.prompts.tuners.scorers import Scorer, LLMJudgeScorer
from aimu.models import OllamaClient, AnthropicClient
from aimu.models.ollama.ollama_client import OllamaModel
from aimu.models.anthropic.anthropic_client import AnthropicModel
from aimu.agents import SimpleAgent, AgenticModelClient
from aimu.tools import MCPClient

nest_asyncio.apply()

DB_PATH = paths.output / "medrxiv.db"
assert DB_PATH.exists(), f"medRxiv vector DB not found at {DB_PATH}. Run scripts/07.2_medrxiv_knowledge_base.py first."

## 02. Benchmark Dataset

Build a small balanced sample from the labeled classification training set. `Benchmark` requires a `content` column (passed into the prompt template's `{content}` placeholder) and accepts an optional `reference` column that scorers can read.

Keep the sample small: every row is a fresh chat for every client, so total cost grows as `rows x clients x tasks`.

In [ ]:
SAMPLE_PER_CLASS = 5

df_all = data.load_classification_training_data().reset_index(drop=True)

df_pos = df_all[df_all.is_modeling].sample(SAMPLE_PER_CLASS, random_state=42)
df_neg = df_all[~df_all.is_modeling].sample(SAMPLE_PER_CLASS, random_state=42)

df_bench = (
    pd.concat([df_pos, df_neg])
    .reset_index(drop=True)
    .rename(columns={"abstract": "content"})
)
df_bench["reference"] = df_bench.is_modeling.map({True: "[YES]", False: "[NO]"})
df_bench[["title", "is_modeling", "reference"]]

## 03. Model Clients and Agent Configurations

Build the slate of clients to compare. Each entry's key becomes the row label in the result DataFrame.

The agent configuration wraps a `SimpleAgent` (with the genscai MCP `search_research_articles` tool) inside an `AgenticModelClient`, which is a `BaseModelClient` that runs the full agentic tool-use loop on every `chat()` call. From `Benchmark`'s perspective it looks like any other client.

Edit the `clients` dict to add or remove configurations. The Ollama models must be pulled locally; the Anthropic client requires `ANTHROPIC_API_KEY`.

In [ ]:
MCP_SERVERS = {
    "mcpServers": {
        "genscai": {"command": "python", "args": [str(paths.package / "tools.py")]},
    }
}

AGENT_SYSTEM = (
    "You are a research assistant. When the user asks about a scientific topic or abstract, "
    "you may call `search_research_articles` once or twice to ground your answer in related "
    "medRxiv literature, then return a concise response. Do not call the tool for simple "
    "yes/no classification questions."
)

agent_inner = OllamaClient(OllamaModel.QWEN_3_5_9B)
agent_inner.mcp_client = MCPClient(MCP_SERVERS)
agent = SimpleAgent(
    model_client=agent_inner,
    name="qwen3.5_9b_agent",
    system_message=AGENT_SYSTEM,
    max_iterations=4,
    reset_messages_on_run=True,
)
agentic_client = AgenticModelClient(agent)

In [ ]:
clients = {
    "ollama:qwen3.5-9b": OllamaClient(OllamaModel.QWEN_3_5_9B),
    "ollama:gemma4-e4b": OllamaClient(OllamaModel.GEMMA_4_E4B),
    "anthropic:haiku-4.5": AnthropicClient(AnthropicModel.CLAUDE_HAIKU_4_5),
    "agent:qwen3.5-9b+mcp": agentic_client,
}
list(clients.keys())

## 04. Task A - Disease Modeling Classification

Reuse the project's classification prompt and define a custom `Scorer` that parses `[YES]`/`[NO]` from the model output and compares it to the labeled `reference`. The score is binary (1.0 / 0.0); `pass_rate` therefore equals accuracy at any `pass_threshold <= 1.0`.

Note: `Benchmark` formats the prompt with `prompt.format(content=row.content)`, so the abstract is passed in as `{content}`.

In [ ]:
CLASSIFICATION_PROMPT = (
    "Read the following scientific paper abstract. Determine if it explicitly refers to or uses "
    "a disease modeling technique - mathematical, statistical, or computational methods used to "
    "simulate, analyze, predict, or interpret disease dynamics.\n\n"
    "Answer [YES] if the abstract describes or references such methods, otherwise [NO]. "
    "Wrap your final answer in square braces. Reply with only [YES] or [NO].\n\n"
    "Abstract:\n{content}"
)


class ExactMatchClassificationScorer(Scorer):
    """Score 1.0 when the parsed [YES]/[NO] matches the reference, else 0.0."""

    def score(self, row) -> tuple[float, str]:
        text = row.output.split("</think>")[-1].lower()
        if "[yes]" in text:
            predicted = "[YES]"
        elif "[no]" in text:
            predicted = "[NO]"
        else:
            return 0.0, f"unparseable output: {row.output[:120]!r}"
        match = predicted == row.reference
        return (1.0 if match else 0.0), f"predicted={predicted} reference={row.reference}"

In [ ]:
classification_bench = Benchmark(
    prompt=CLASSIFICATION_PROMPT,
    data=df_bench,
    scorer=ExactMatchClassificationScorer(),
    pass_threshold=1.0,
    generate_kwargs={"max_new_tokens": 8, "temperature": 0.01},
)

classification_results = classification_bench.run(clients)
classification_results.metrics

## 05. Task B - Abstract Summarization

Each client must turn the abstract into a 2-3 sentence plain-English summary. Outputs are graded by an `LLMJudgeScorer` backed by Claude Haiku - the same judge across all clients so scores are comparable.

This is the task where the `agent:qwen3.5-9b+mcp` configuration can plausibly outperform its base model, since it can search related medRxiv abstracts before writing the summary.

In [ ]:
judge_client = AnthropicClient(AnthropicModel.CLAUDE_HAIKU_4_5)

summary_scorer = LLMJudgeScorer(
    judge_client=judge_client,
    criteria=(
        "A good summary is 2-3 sentences, faithful to the abstract (no hallucinated findings or "
        "methods), names the disease and the modeling or analytical approach when present, and "
        "is in plain English a non-specialist can follow."
    ),
)

SUMMARY_PROMPT = (
    "Summarize the following scientific abstract in 2-3 plain-English sentences. "
    "Cover the disease studied, the methods or model used, and the key finding. "
    "Do not invent details that are not in the abstract.\n\n"
    "Abstract:\n{content}"
)

In [ ]:
summary_bench = Benchmark(
    prompt=SUMMARY_PROMPT,
    data=df_bench,
    scorer=summary_scorer,
    pass_threshold=0.7,
)

summary_results = summary_bench.run(clients)
summary_results.metrics

## 06. Side-by-Side Comparison

Stack both tasks so each client has one row per (task, metric) for direct comparison.

In [ ]:
combined = pd.concat(
    {
        "classification": classification_results.metrics,
        "summarization": summary_results.metrics,
    },
    axis=1,
)
combined

## 07. Save Results

`BenchmarkResults` carries the prompt that was used, so re-runs are reproducible from the saved JSON. Persist both task runs to `output/`.

In [ ]:
classification_results.to_json(str(paths.output / "benchmark_classification.json"))
summary_results.to_json(str(paths.output / "benchmark_summarization.json"))
combined.to_csv(paths.output / "benchmark_combined.csv")

## 08. Cleanup

Drop clients to release Ollama keep-alive slots and free GPU memory before running another notebook against the same device.

In [ ]:
for client in clients.values():
    del client
del clients, agentic_client, agent, agent_inner, judge_client